# **SNOWFLAKE CORTEX COMPLETE FINANCIAL SERVICES DEMO**

## Authors: John Heisler, Garrett Frere

In this demo, using Snowflake Cortex (https://www.snowflake.com/en/data-cloud/cortex/), we will build an AI-infused Data Pipeline with Cortex Complete.

### AI Pipeline Overview

We'll learn how to extract raw text from a PDF, perform prompt engineering, and pass custom prompts and data to a large language model of our choosing all without leaving Snowflake.

Specifically, we will be taking on the role of an AI Engineer who is working closely with a portfolio team at an asset manager. The portfolio team would like to speed up their ingestion and comprehension of statements by the Federal Open Market Committee (FOMC) who determines the direction of monetary policy by directing open market operations. Ultimately they would like to get a signal as to whether interest rates will increase, remain the same, or increase (hawkish, or, dovish respectively).

I refer to this as an AI pipeline because we can imbue this type of signal generation with AI much further up the data delivery value chain. In this way, we will maximize the value of our work imbuing into a common dataset. End users will not need invoke any additional logic; good design is invisible!

### Next Steps

 * To industrialize this demo with continuous ingestion and scoring, please check out the `FSI_Cortex_AI_Pipeline_Industrialization.ipynb` notebook in this repository
 * Check out the companion demo in this repository: `FSI_Cortex_Search.ipynb`

# 🛑 BEFORE YOU START 🛑

**Be sure to do the following FIRST to create dependent database objects for the following steps**:
1. Run the `1_SQL_SETUP_FOMC.sql` script
2. Load the PDF docs in the `FOMC_DOCS` directory

------

### AI Pipeline: Step 1 - Extract Text from PDFs using AI_PARSE_DOCUMENT

We use Snowflake's native `AI_PARSE_DOCUMENT` function to extract text from PDFs. This replaces the need for custom Python UDFs with PyPDF2.

> **Note:** `AI_PARSE_DOCUMENT` provides better layout preservation, table extraction, and requires no external packages.

In [ ]:
USE ROLE SYSADMIN;

USE DATABASE GEN_AI_FSI;
USE SCHEMA FOMC;

In [ ]:
-- AI_PARSE_DOCUMENT is a native SQL function - no UDF registration needed!
-- Test the function on a sample file from the stage:

SELECT 
    relative_path,
    AI_PARSE_DOCUMENT(
        TO_FILE('@gen_ai_fsi.fomc.fed_pdf', relative_path),
        {'mode': 'LAYOUT'}
    ):content::VARCHAR AS extracted_text
FROM directory(@gen_ai_fsi.fomc.fed_pdf)
LIMIT 1;

### AI Pipeline: Step 2 - Define Structured Output Schema

With `AI_COMPLETE` structured outputs, we no longer need to ask the LLM to return JSON - we can enforce the exact schema. The prompt focuses on the task, and the schema guarantees consistent output.

In [ ]:
-- Define the prompt as a SQL function (simpler than Python UDF)
CREATE OR REPLACE FUNCTION gen_ai_fsi.fomc.generate_prompt(document_text VARCHAR)
RETURNS VARCHAR
LANGUAGE SQL
AS
$$
    '<Role> You are an experienced Senior Economist deeply knowledgeable on Federal Reserve guidance including FOMC or Federal Open Market Committee meeting minutes and communications.
    You are an expert in interpreting Hawkish and Dovish signals from the Fed or Federal Reserve. Such signals are derived from guidance conveyed in FOMC meeting notes and communications.
    
    As an analyst, you excel at discerning macroeconomic trends for each FOMC meeting notes and communications published by the Federal Reserve.
    The signal or trends are either Hawkish or Dovish based on the growth outlook and inflation outlook of the Fed. The Federal Reserve has a long 
    term objective of keeping inflation around 2%, and low unemployment. Hawkish sentiment could imply 
    the Federal Reserve intends to raise interest rates to increase the cost of borrowing and slow economic activity. 
    The Fed typically increases interest rates when inflation is high or rising, or when the unemployment 
    rate is low or falling. Conversely, dovish sentiment could imply the Federal Reserve intends to lower interest 
    rates to allow easier access borrowing and lowering the cost of money to stimulate economic activity. The Fed 
    typically decreases interest rates when inflation is low or falling, or when the unemployment rate is high or rising.
    
    Signal categories known as Economic Policy Stances:
    Hawkish stance: characterized by a focus on combating inflation, advocating for higher interest rates, tolerant to higher levels of unemployment.
    Dovish stance: characterized by a focus on stimulating economic growth, reducing unemployment, tolerant to higher levels of inflation.
    Neutral stance: characterized by balance between combating inflation and supporting economic growth.
    </Role>
    
    <Data> 
    You are provided the text of a Federal Reserve Guidance or FOMC meeting notes as context. 
    </Data>

    <FOMC_meeting_notes>
    ' || document_text || '
    </FOMC_meeting_notes>
    
    <Task>
    1) Review the provided FOMC communication or meeting notes text.
    2) Consider the FOMC members tone and sentiment around economic conditions.
    3) Consider specific guidance that validates the tone concerning macro economic conditions.
    4) Classify if the FOMC communication indicates Hawkish, Dovish, or Neutral outlook. Be critical - do not categorize as Neutral unless necessary.
    5) Summarize a concise rationale for your classification.
    </Task>'
$$;

### AI Pipeline: Step 3 - Ingest Text and Determine Signal

Now we use native Snowflake AI functions in a simple INSERT statement:
* `AI_PARSE_DOCUMENT` extracts text from PDFs
* `AI_COMPLETE` with structured output guarantees valid JSON response

In [ ]:
INSERT INTO gen_ai_fsi.fomc.pdf_full_text (id, relative_path, size, last_modified, md5, etag, file_url, file_text, file_date, sentiment)
WITH cte AS (
    SELECT 
        gen_ai_fsi.fomc.fed_pdf_full_text_sequence.nextval AS id,
        relative_path,
        size,
        last_modified,
        md5,
        etag,
        file_url,
        -- Use AI_PARSE_DOCUMENT instead of custom UDF
        AI_PARSE_DOCUMENT(
            TO_FILE('@gen_ai_fsi.fomc.fed_pdf', relative_path),
            {'mode': 'LAYOUT'}
        ):content::VARCHAR AS file_text,
        TRY_TO_DATE(REGEXP_SUBSTR(relative_path, '\\d{8}'), 'YYYYMMDD') AS file_date
    FROM gen_ai_fsi.fomc.fomc_stream
    WHERE metadata$action = 'INSERT'
)
SELECT 
    id,
    relative_path,
    size,
    last_modified,
    md5,
    etag,
    file_url,
    file_text,
    file_date,
    -- Use AI_COMPLETE with structured output for guaranteed JSON
    AI_COMPLETE(
        'mistral-large2',
        gen_ai_fsi.fomc.generate_prompt(file_text),
        response_format => {
            'type': 'json',
            'schema': {
                'type': 'object',
                'properties': {
                    'Signal': {
                        'type': 'string',
                        'enum': ['Hawkish', 'Dovish', 'Neutral']
                    },
                    'Signal_Summary': {
                        'type': 'string'
                    }
                },
                'required': ['Signal', 'Signal_Summary']
            }
        }
    ) AS sentiment
FROM cte;

### AI Pipeline: Step 3.1 - Check out the result

select from our PDF table to view our signal and a summary or reasoning.

In [ ]:
select * from GEN_AI_FSI.FOMC.PDF_FULL_TEXT;

-------

## Build a RAG Interface on FOMC Documents

Awesome, we have created the pipeline to ingest and generate a signal when new data is avaiable -- this is our **AI pipeline**. 

Next, let's also give our users a means to ask more detailed questions about the content in the documents with a RAG interface. We'll use Cortex Search and the data we already have in the Stage as a foundation. 

In this section we'll: 
1. Create a new table function to chunk the pdfs.
2. Use the chunking function to break the text into chunks and load it into our table.
3. Create a Cortex Search Service to handle the vectorization and search functionality.
4. Create a Chat interface with Streamlit.

## Chunking with Native Function

We use Snowflake's built-in `SPLIT_TEXT_RECURSIVE_CHARACTER` function instead of a custom LangChain UDTF. This is simpler, faster, and requires no external packages. 

In [ ]:
-- No custom function needed! We'll use SNOWFLAKE.CORTEX.SPLIT_TEXT_RECURSIVE_CHARACTER
-- directly in the INSERT statement below.

-- Preview how the chunking works on existing data:
SELECT 
    id,
    relative_path,
    c.value::VARCHAR AS chunk
FROM gen_ai_fsi.fomc.pdf_full_text,
     LATERAL FLATTEN(
         SNOWFLAKE.CORTEX.SPLIT_TEXT_RECURSIVE_CHARACTER(
             file_text,
             'none',      -- separator type
             500,         -- chunk_size (tokens)
             50           -- chunk_overlap (tokens)
         )
     ) c
LIMIT 10;

## Build the Chunk Table

Using the native `SPLIT_TEXT_RECURSIVE_CHARACTER` function, we chunk all documents from our full text table.

In [ ]:
TRUNCATE TABLE gen_ai_fsi.fomc.pdf_chunks;

INSERT INTO gen_ai_fsi.fomc.pdf_chunks (id, full_text_fk, relative_path, file_date, chunk)
SELECT 
    gen_ai_fsi.fomc.fed_pdf_chunk_sequence.nextval AS id,
    pft.id AS full_text_fk,
    pft.relative_path,
    pft.file_date,
    REPLACE(c.value::VARCHAR, '''', '') AS chunk
FROM gen_ai_fsi.fomc.pdf_full_text pft,
     LATERAL FLATTEN(
         SNOWFLAKE.CORTEX.SPLIT_TEXT_RECURSIVE_CHARACTER(
             pft.file_text,
             'none',
             500,
             50
         )
     ) c;

## Create a Cortex Search Service

Cortex Search enables low-latency, high-quality “fuzzy” search over your Snowflake data. Cortex Search powers a broad array of search experiences for Snowflake users including Retrieval Augmented Generation (RAG) applications leveraging Large Language Models (LLMs).

We'll use this service later to power our RAG application. 

In [ ]:
-- Create Cortex Search Service with PRIMARY KEY and explicit embedding model
CREATE OR REPLACE CORTEX SEARCH SERVICE SRCH_FED
ON CHUNK
PRIMARY KEY (ID)
ATTRIBUTES FILE_DATE
WAREHOUSE = GEN_AI_FSI_WH
TARGET_LAG = '1 day'
EMBEDDING_MODEL = 'snowflake-arctic-embed-l-v2.0'
AS (
    SELECT 
        ID::VARCHAR AS ID,
        FILE_DATE::STRING AS FILE_DATE,
        CHUNK
    FROM PDF_CHUNKS
);

# FOMC Chat Interface
* We're leveraging our Cortex Search service enabling users to ask targeted questions of the documents in their stage.
* a robust chat interface could be built to handle this, for the demo, we have a bare bones interaction.

In [ ]:
from snowflake.snowpark.context import get_active_session
from snowflake.core import Root
import streamlit as st
import json

# Get our session
session = get_active_session()

# Access search service through Python API
root = Root(session)
search_service = (root
                  .databases["GEN_AI_FSI"]
                  .schemas["FOMC"]
                  .cortex_search_services["SRCH_FED"]    
)

# Create a function to generate response using TRY_AI_COMPLETE
def complete_cs(model_name, prompt):
    # Escape single quotes in prompt for SQL
    safe_prompt = prompt.replace("'", "''")
    cmd = f"""SELECT TRY_AI_COMPLETE('{model_name}', '{safe_prompt}') AS response"""
    df_response = session.sql(cmd).collect()
    response = df_response[0].RESPONSE
    return response

# Get FOMC files
database = 'GEN_AI_FSI'
schema = 'FOMC'

# USER INPUT: select time frame
query_document_dates = f"""SELECT DISTINCT FILE_DATE FROM {database}.{schema}.PDF_CHUNKS ORDER BY FILE_DATE DESC;"""
df_document_dates = session.sql(query_document_dates).to_pandas()

# USER INPUT: select model
query_models = f"""SELECT MODEL FROM {database}.{schema}.MODELS"""
df_models = session.sql(query_models).to_pandas()

# USER INPUT: display
col1, col2 = st.columns(2)
with col1:
    user_input_date = st.selectbox("Select Document Date", df_document_dates, key="CS_date_select_box")
with col2: 
    user_input_model = st.selectbox("Select Model", df_models, key="CS_model_select_box")

# Generate a response
user_input_question = st.text_input("Ask me a question")

ask = st.button("Ask", key="button_ask")
if ask: 
    # Get the chunks that are relevant to the question
    response = search_service.search(
        user_input_question,
        columns=["ID", "FILE_DATE", "CHUNK"],
        filter={"@eq": {"FILE_DATE": str(user_input_date)}},
        limit=5,
    )
    
    # Convert response to dict
    results = json.loads(response.to_json())
    
    # Transform the data into a single string
    context_full = ""
    for chunk in results.get('results', []):
        context_full += chunk['CHUNK'] + " "

    # Build our prompt
    cs_prompt = f'''Role: You are an expert Senior Economist deeply knowledgeable on Federal Reserve documents and guidance including FOMC or Federal Open Market Committee meeting minutes and communications. You are an expert in interpreting and answering investment-related questions based on meeting minutes and communications which you are provided as context with each question.
            
Data: You are provided with relevant text of a Federal Reserve Guidance or FOMC meeting notes relevant to the question asked. These meeting notes are generally released before the Federal Reserve takes action on economic policy.
            
Task: Follow these instructions:
1) Answer the question based on the context. 
2) Be terse and do not consider information outside what you have been provided in the question and context.
            
Output: Produce thorough, valid, grammatically correct, and concise response in a professional tone. Please do not preface your response. Also provide the document and location you used to answer the question.

Context: {context_full}

Question: Based on documents released on {user_input_date}, {user_input_question}'''

    data = complete_cs(user_input_model, cs_prompt)
    
    with st.chat_message("model", avatar="assistant"):
        st.write(data)